# Calibration of the implied volatility of a european call for the Black-Scholes Model through MARL

## We first need to set up some functions for the Black-Scholes equation, and for calculating implied volatility based on actual price of the option

In [18]:
torch.randint(0, 3, (5,))

tensor([1, 1, 1, 2, 2])

In [1]:
import math
from scipy.stats import norm
from scipy.optimize import brentq

def black_scholes_call_price(S, K, T, r, sigma):
    """
    Compute the Black-Scholes price for a European call option.
    
    Parameters:
        S (float): current underlying price
        K (float): strike price
        T (float): time to expiration (in years)
        r (float): risk-free interest rate (annualized)
        sigma (float): volatility of the underlying asset
    
    Returns:
        float: call option price according to Black-Scholes
    """
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    call = S * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
    return call

def implied_volatility_call(option_price, S, K, T, r):
    """
    Compute the implied volatility for a European call option given its market price.
    
    Parameters:
        option_price (float): market price of the option
        S (float): current underlying price
        K (float): strike price
        T (float): time to expiration (in years)
        r (float): risk-free interest rate (annualized)
    
    Returns:
        float: implied volatility
    """
    # Define the objective function whose root we want to find:
    # f(sigma) = BS_call_price(sigma) - option_price = 0.
    def objective(sigma):
        return black_scholes_call_price(S, K, T, r, sigma) - option_price
    
    # Use Brent's method to solve for sigma in a reasonable interval.
    # Volatility is positive so we bracket the solution between 1e-6 and, say, 5.0.
    implied_vol = brentq(objective, 1e-6, 5.0)
    return implied_vol

### Example:

In [16]:
S = 100       # Current stock price
K = 100       # Strike price
T = 0.98    # Time to expiration in years
r = 0.05      # Risk-free interest rate (5%)
sigma_true = 0.2  # True volatility
# Compute option price from the true volatility
option_price = black_scholes_call_price(S, K, T, r, sigma_true)
print("Market Option Price:", option_price)

# Now compute the implied volatility from the market price
iv = implied_volatility_call(option_price, S, K, T, r)
print("Implied Volatility:", iv)


Market Option Price: 10.321879318528453
Implied Volatility: 0.2000000000000002


## We now build the reward function

In [17]:
def reward(simulated_price:float, market_price:float, option_params:dict):
    implied_vol_simulated = implied_volatility_call(simulated_price, 
                                                    option_params['S'], 
                                                    option_params['K'], 
                                                    option_params['T'], 
                                                    option_params['r'])
    implied_vol_market = implied_volatility_call(market_price,
                                                 option_params['S'], 
                                                 option_params['K'], 
                                                 option_params['T'], 
                                                 option_params['r'])
    return -(implied_vol_simulated - implied_vol_market)**2

In [18]:
option_params = {'S': 100,
                 'K': 100,
                 'T': 1,
                 'r': 0.05}

In [21]:
reward(9, 10.45, option_params)

-0.001513136307222987

# Set up Actor Network

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

class ActorNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=64):
        super(ActorNetwork, self).__init__()
        # Simple 2-layer MLP for policy
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
            nn.Softmax(dim=-1)  # output probabilities over actions
        )
    
    def forward(self, obs):
        # obs is a torch tensor of shape (obs_dim,) or (batch, obs_dim)
        return self.net(obs)  # returns action probabilities

# Set up Critic Network

In [4]:
# Critic network: state-value function for one agent (could use global state)
class CriticNetwork(nn.Module):
    def __init__(self, state_dim, hidden_dim=64):
        super(CriticNetwork, self).__init__()
        # Simple 2-layer MLP for value function
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)  # outputs a single value
        )
    
    def forward(self, state):
        # state can be global state or observation (depending on centralization)
        return self.net(state)  # returns value (scalar) as tensor

# Create Agent

In [5]:
class Agent:
    def __init__(self, obs_dim, act_dim, state_dim=None, lr=1e-3):
        """
        obs_dim: dimension of the agent's observation space
        act_dim: number of possible actions for the agent
        state_dim: dimension of state input for critic. If None, use obs_dim.
                   (In a centralized critic scenario, state_dim could be larger than obs_dim)
        """
        if state_dim is None:
            state_dim = obs_dim
        # Initialize actor and critic networks
        self.actor = ActorNetwork(obs_dim, act_dim)
        self.critic = CriticNetwork(state_dim)
        # Optimizers for actor and critic
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr)
    
    def select_action(self, obs):
        """
        Choose an action for the agent given its observation.
        obs: can be a numpy array or torch tensor of shape (obs_dim,)
        Returns: action (int), log_probability of that action
        """
        obs_t = torch.tensor(obs, dtype=torch.float32)  # convert observation to tensor
        # Get action probabilities from the actor
        probs = self.actor(obs_t)             # shape (act_dim,)
        dist = Categorical(probs)             # create a categorical distribution
        action = dist.sample()               # sample an action
        log_prob = dist.log_prob(action)     # log probability of the chosen action
        return action.item(), log_prob  # return action as int and log-prob (as tensor)
    
    def evaluate_action(self, obs, action):
        """
        Compute log-probability of a given action under the current policy (for training).
        """
        obs_t = torch.tensor(obs, dtype=torch.float32)
        probs = self.actor(obs_t)
        dist = Categorical(probs)
        action_t = torch.tensor(action, dtype=torch.int64)
        return dist.log_prob(action_t)  # tensor of log-prob

In [6]:
# Example cooperative environment (for demonstration purposes)
class SimpleCoopEnv:
    """
    A simple multi-agent environment for cooperative tasks.
    In this toy example, we have `N` agents each choosing between two actions (0 or 1).
    The environment rewards the team if all agents choose the same action at a timestep.
    Each episode lasts for a fixed number of steps.
    """
    def __init__(self, n_agents=2, ep_length=5):
        self.n_agents = n_agents
        self.ep_length = ep_length
        self.step_count = 0
        # Observation: here just a dummy vector (all zeros) for each agent
        self.obs_dim = 1
        # Action space: 2 discrete actions (0 or 1) for each agent
        self.act_dim = 2
    
    def reset(self):
        self.step_count = 0
        # Each agent gets an observation (here just 0.0 as a placeholder)
        obs = [np.zeros(self.obs_dim, dtype=np.float32) for _ in range(self.n_agents)]
        return obs
    
    def step(self, actions):
        # actions: list of actions (ints) of length n_agents
        self.step_count += 1
        # Check if all actions are the same (cooperative success condition)
        reward = 1.0 if all(a == actions[0] for a in actions) else 0.0
        done = (self.step_count >= self.ep_length)
        # Next observation (for simplicity, just zeros again)
        obs_next = [np.zeros(self.obs_dim, dtype=np.float32) for _ in range(self.n_agents)]
        # In cooperative tasks, we can give all agents the same reward
        rewards = [reward] * self.n_agents
        return obs_next, rewards, done, {}

In [7]:
# Training function for multi-agent actor-critic
def train_multi_agent(env, agents, num_episodes=1000, gamma=0.99):
    """
    env: multi-agent environment with methods reset() and step(actions).
         - reset() -> list of initial observations (one per agent).
         - step(actions) -> (next_obs_list, reward_list, done, info)
    agents: list of Agent instances (one per agent).
    num_episodes: number of episodes to train.
    gamma: discount factor.
    """
    n_agents = len(agents)
    for episode in range(num_episodes):
        # Reset environment and get initial observations for all agents
        obs_n = env.reset()  # list of observations, length = n_agents
        # To store trajectory data for this episode
        log_probs_n = [[] for _ in range(n_agents)]  # log probabilities of actions for each agent
        rewards_n = [[] for _ in range(n_agents)]    # rewards for each agent (could also use one list since shared)
        values_n = [[] for _ in range(n_agents)]     # critic value estimates
        done = False

        # Step through the episode
        while not done:
            actions = []
            current_values = []
            log_probs = []
            # Each agent selects an action
            for i, agent in enumerate(agents):
                action, log_prob = agent.select_action(obs_n[i])
                actions.append(action)
                log_probs.append(log_prob)
                # For critic, get the value of current state. Use global state if needed.
                # Here we use the agent's own observation as input to critic.
                state_input = torch.tensor(obs_n[i], dtype=torch.float32)
                value = agent.critic(state_input)
                current_values.append(value)
            # Perform the joint action in the environment
            next_obs_n, reward_n, done, _ = env.step(actions)
            # Store log_probs, rewards, and values
            for i in range(n_agents):
                log_probs_n[i].append(log_probs[i])
                rewards_n[i].append(reward_n[i])
                values_n[i].append(current_values[i])
            # Move to next state
            obs_n = next_obs_n

        # Episode is done, now compute returns and update networks
        # We assume all agents have the same episode length and termination (since it's a shared environment).
        # Compute discounted returns for each agent
        returns_n = [[] for _ in range(n_agents)]
        # Start from last timestep and go backwards to compute G_t
        for i in range(n_agents):
            G = 0.0
            for reward in reversed(rewards_n[i]):
                G = reward + gamma * G
                returns_n[i].insert(0, G)  # insert at front
        # Convert lists to tensors for easier math
        for i in range(n_agents):
            # Stack values and returns for agent i
            returns = torch.tensor(returns_n[i], dtype=torch.float32)
            values = torch.stack(values_n[i]).squeeze(-1)  # make it 1D (shape = episode_len)
            # Compute advantage estimates
            advantages = returns - values.detach()  # detach critic values to avoid affecting critic grads
            # Update actor (policy) for agent i
            agent = agents[i]
            agent.actor_optimizer.zero_grad()
            # Stack log_probs list into tensor
            log_probs = torch.stack(log_probs_n[i])
            actor_loss = -(log_probs * advantages).mean()
            actor_loss.backward()
            agent.actor_optimizer.step()

            # Update critic (value function) for agent i
            agent.critic_optimizer.zero_grad()
            # Critic loss: MSE of returns vs. predicted values
            critic_loss = nn.functional.mse_loss(values, returns)
            critic_loss.backward()
            agent.critic_optimizer.step()
        # (Optional) Print training progress
        if (episode+1) % 100 == 0:
            avg_return = np.mean([returns_n[0][0] for _ in range(n_agents)])  # return from first timestep as episode return
            print(f"Episode {episode+1}/{num_episodes}, Avg team return: {avg_return:.2f}")

In [11]:
# Create a cooperative environment with 2 agents
env = SimpleCoopEnv(n_agents=8, ep_length=5)
n_agents = env.n_agents
obs_dim = env.obs_dim    # each agent's observation dimension
act_dim = env.act_dim    # each agent's action space size

# Initialize agents
agents = [Agent(obs_dim, act_dim) for _ in range(n_agents)]

# Train the agents in the environment
train_multi_agent(env, agents, num_episodes=600, gamma=0.95)

# Test the learned policies
obs = env.reset()
done = False
total_reward = 0
print("Testing learned policies:")
while not done:
    actions = []
    for i, agent in enumerate(agents):
        # Each agent selects the argmax action (most likely action) for testing deterministically
        obs_t = torch.tensor(obs[i], dtype=torch.float32)
        probs = agent.actor(obs_t)
        action = int(torch.argmax(probs).item())
        actions.append(action)
    obs, rewards, done, _ = env.step(actions)
    total_reward += rewards[0]  # all rewards are same
    print(f"Actions: {actions}, Reward: {rewards[0]}")
print(f"Total reward in test episode: {total_reward}")


Episode 100/600, Avg team return: 0.00
Episode 200/600, Avg team return: 0.00
Episode 300/600, Avg team return: 4.52
Episode 400/600, Avg team return: 4.52
Episode 500/600, Avg team return: 4.52
Episode 600/600, Avg team return: 3.67
Testing learned policies:
Actions: [0, 0, 0, 0, 0, 0, 0, 0], Reward: 1.0
Actions: [0, 0, 0, 0, 0, 0, 0, 0], Reward: 1.0
Actions: [0, 0, 0, 0, 0, 0, 0, 0], Reward: 1.0
Actions: [0, 0, 0, 0, 0, 0, 0, 0], Reward: 1.0
Actions: [0, 0, 0, 0, 0, 0, 0, 0], Reward: 1.0
Total reward in test episode: 5.0
